# 10 - PySpark Performance Optimization Notebook

PySpark performance optimization is about reducing unnecessary data movement, controlling partition sizes, using memory intentionally, and helping Spark choose efficient physical plans.

In this notebook, you will learn:

- How to create a SparkSession with practical optimization defaults
- How partition count affects task overhead and write performance
- When and how to cache DataFrames safely
- How broadcast joins avoid expensive shuffles
- Why filters and column selection should happen early
- How to tune shuffle partitions for your cluster
- How to inspect execution plans using `explain(True)`
- A quick reference checklist for common optimization patterns

## Step 1 — SparkSession setup

This notebook targets your local Spark Docker cluster, so it uses `spark://spark-master:7077`.

The settings below are intentionally conservative for a small local cluster. In production, tune them based on executor count, cores, memory, and workload size.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, broadcast, spark_partition_id
import os
import shutil
import time

spark = (
    SparkSession.builder
    .appName("10-PySpark-Performance-Optimization")
    .master("spark://spark-master:7077")
    .config("spark.sql.shuffle.partitions", "12")
    .config("spark.sql.autoBroadcastJoinThreshold", "10485760")
    .config("spark.default.parallelism", "12")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Spark application id:", spark.sparkContext.applicationId)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("Broadcast threshold:", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))
print("Default parallelism:", spark.sparkContext.defaultParallelism)

Spark version: 4.0.2
Spark application id: app-20260425204917-0011
Shuffle partitions: 12
Broadcast threshold: 10485760
Default parallelism: 12


## Step 2 — Helper functions

We will use small helper functions for timing and safe output cleanup.

In [2]:
BASE_PATH = "/workspace/data/performance_optimization"
GOOD_OUTPUT = f"{BASE_PATH}/good_output"
BAD_OUTPUT = f"{BASE_PATH}/bad_output"

def safe_rm(path):
    if os.path.exists(path):
        shutil.rmtree(path)
        print(f"Removed: {path}")

def timed(label, fn):
    start = time.perf_counter()
    result = fn()
    elapsed = time.perf_counter() - start
    print(f"{label}: {elapsed:.3f} sec")
    return result

safe_rm(BASE_PATH)

## Step 3 — Partition strategy

Goal: target approximately 128–256 MB per partition for large file outputs.

Too many partitions create scheduler overhead and many small files. Too few partitions create long-running tasks and poor parallelism.

For this tutorial, the dataset is smaller than a real 1.5 GB dataset, but the pattern is the same.

In [3]:
df_large = (
    spark.range(1_000_000)
    .withColumn("value", col("id") % 100)
)

print("Initial partitions:", df_large.rdd.getNumPartitions())

optimized_df = df_large.repartition(15)

print("Partitions after repartition:", optimized_df.rdd.getNumPartitions())

optimized_df.groupBy(spark_partition_id().alias("partition_id")).count().orderBy("partition_id").show()

Initial partitions: 12


[Stage 0:======================================>                   (8 + 4) / 12]

Partitions after repartition: 15


[Stage 3:=========================================>               (11 + 4) / 15]

+------------+-----+
|partition_id|count|
+------------+-----+
|           0|66667|
|           1|66667|
|           2|66665|
|           3|66667|
|           4|66667|
|           5|66666|
|           6|66668|
|           7|66667|
|           8|66665|
|           9|66667|
|          10|66667|
|          11|66668|
|          12|66666|
|          13|66666|
|          14|66667|
+------------+-----+



## Step 4 — Write optimized output

Writing after an intentional repartition gives you more control over file count.

In real workloads, choose partition count based on expected output size and target file size.

In [4]:
timed(
    "Write optimized Parquet output",
    lambda: optimized_df.write.mode("overwrite").parquet(GOOD_OUTPUT)
)

print("Output path:", GOOD_OUTPUT)

Write optimized Parquet output: 3.315 sec
Output path: /workspace/data/performance_optimization/good_output


## Step 5 — Strategic caching

Goal: cache only DataFrames reused two or more times.

Caching has a cost: Spark must compute and store the DataFrame. It is useful only when reuse saves more time than the cache cost.

Always call `unpersist()` when the cached DataFrame is no longer needed.

In [5]:
df_cached = spark.range(500_000).toDF("id").cache()

timed("First count triggers computation and cache fill", lambda: df_cached.count())
timed("Second action can reuse cached data", lambda: df_cached.filter(col("id") > 250_000).count())

df_cached.unpersist()
print("Cache released.")

First count triggers computation and cache fill: 0.589 sec
Second action can reuse cached data: 0.287 sec
Cache released.


## Step 6 — Broadcast small tables in joins

Goal: avoid expensive shuffle joins when one side is small.

A broadcast join sends the small table to every executor and allows the large table to be scanned without shuffling both sides.

In [6]:
df_small = (
    spark.range(100)
    .withColumn("category", col("id") % 5)
)

result = df_large.join(broadcast(df_small), on="id", how="inner")

result.explain(True)

print("Join result count:", result.count())

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [id])
:- Project [id#0L, (id#0L % cast(100 as bigint)) AS value#1L]
:  +- Range (0, 1000000, step=1, splits=Some(12))
+- ResolvedHint (strategy=broadcast)
   +- Project [id#84L, (id#84L % cast(5 as bigint)) AS category#85L]
      +- Range (0, 100, step=1, splits=Some(12))

== Analyzed Logical Plan ==
id: bigint, value: bigint, category: bigint
Project [id#0L, value#1L, category#85L]
+- Join Inner, (id#0L = id#84L)
   :- Project [id#0L, (id#0L % cast(100 as bigint)) AS value#1L]
   :  +- Range (0, 1000000, step=1, splits=Some(12))
   +- ResolvedHint (strategy=broadcast)
      +- Project [id#84L, (id#84L % cast(5 as bigint)) AS category#85L]
         +- Range (0, 100, step=1, splits=Some(12))

== Optimized Logical Plan ==
Project [id#0L, value#1L, category#85L]
+- Join Inner, (id#0L = id#84L), rightHint=(strategy=broadcast)
   :- Project [id#0L, (id#0L % 100) AS value#1L]
   :  +- Range (0, 1000000, step=1, splits=Some(12))
   +- Project [

## Step 7 — Push filters early

Goal: reduce data volume before joins, aggregations, and writes.

Catalyst can often push filters down automatically, but writing transformations clearly makes the optimization intent obvious and avoids unnecessary intermediate data.

In [7]:
filtered_small = df_small.filter(col("category") == 2)

good_result = df_large.join(filtered_small, on="id", how="inner")

cols_needed = ["id", "value", "category"]

good_result.select(*cols_needed).show(5)
good_result.select(*cols_needed).explain(True)

+---+-----+--------+
| id|value|category|
+---+-----+--------+
|  2|    2|       2|
|  7|    7|       2|
| 12|   12|       2|
| 17|   17|       2|
| 22|   22|       2|
+---+-----+--------+
only showing top 5 rows
== Parsed Logical Plan ==
'Project ['id, 'value, 'category]
+- Project [id#0L, value#1L, category#85L]
   +- Join Inner, (id#0L = id#84L)
      :- Project [id#0L, (id#0L % cast(100 as bigint)) AS value#1L]
      :  +- Range (0, 1000000, step=1, splits=Some(12))
      +- Filter (category#85L = cast(2 as bigint))
         +- Project [id#84L, (id#84L % cast(5 as bigint)) AS category#85L]
            +- Range (0, 100, step=1, splits=Some(12))

== Analyzed Logical Plan ==
id: bigint, value: bigint, category: bigint
Project [id#0L, value#1L, category#85L]
+- Project [id#0L, value#1L, category#85L]
   +- Join Inner, (id#0L = id#84L)
      :- Project [id#0L, (id#0L % cast(100 as bigint)) AS value#1L]
      :  +- Range (0, 1000000, step=1, splits=Some(12))
      +- Filter (category#85L

## Step 8 — Select only needed columns early

Wide DataFrames increase serialization, shuffle, and memory pressure.

Selecting only required columns before joins or aggregations reduces the amount of data Spark must move and process.

In [8]:
wide_df = (
    df_large
    .withColumn("extra_1", col("id") * 10)
    .withColumn("extra_2", col("id") * 20)
    .withColumn("extra_3", col("id") * 30)
)

narrow_df = wide_df.select("id", "value")

print("Wide columns:", wide_df.columns)
print("Narrow columns:", narrow_df.columns)

narrow_df.groupBy("value").count().explain(True)

Wide columns: ['id', 'value', 'extra_1', 'extra_2', 'extra_3']
Narrow columns: ['id', 'value']
== Parsed Logical Plan ==
'Aggregate ['value], ['value, 'count(1) AS count#106]
+- Project [id#0L, value#1L]
   +- Project [id#0L, value#1L, extra_1#103L, extra_2#104L, (id#0L * cast(30 as bigint)) AS extra_3#105L]
      +- Project [id#0L, value#1L, extra_1#103L, (id#0L * cast(20 as bigint)) AS extra_2#104L]
         +- Project [id#0L, value#1L, (id#0L * cast(10 as bigint)) AS extra_1#103L]
            +- Project [id#0L, (id#0L % cast(100 as bigint)) AS value#1L]
               +- Range (0, 1000000, step=1, splits=Some(12))

== Analyzed Logical Plan ==
value: bigint, count: bigint
Aggregate [value#1L], [value#1L, count(1) AS count#106L]
+- Project [id#0L, value#1L]
   +- Project [id#0L, value#1L, extra_1#103L, extra_2#104L, (id#0L * cast(30 as bigint)) AS extra_3#105L]
      +- Project [id#0L, value#1L, extra_1#103L, (id#0L * cast(20 as bigint)) AS extra_2#104L]
         +- Project [id#0L, va

## Step 9 — Tune shuffle partitions

Goal: match shuffle partitions to cluster capacity and data volume.

The Spark default of 200 is often too high for small local clusters and too low for very large workloads.

A practical starting point is 2x–4x the available cores, then validate with Spark UI and task duration.

In [9]:
default_parallelism = spark.sparkContext.defaultParallelism
target_partitions = max(default_parallelism * 3, 4)

spark.conf.set("spark.sql.shuffle.partitions", str(target_partitions))

print("Default parallelism:", default_parallelism)
print("Updated shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

agg_result = df_large.groupBy("value").count()

agg_result.explain(True)
agg_result.orderBy("value").show(10)

Default parallelism: 12
Updated shuffle partitions: 36
== Parsed Logical Plan ==
'Aggregate ['value], ['value, 'count(1) AS count#112]
+- Project [id#0L, (id#0L % cast(100 as bigint)) AS value#1L]
   +- Range (0, 1000000, step=1, splits=Some(12))

== Analyzed Logical Plan ==
value: bigint, count: bigint
Aggregate [value#1L], [value#1L, count(1) AS count#112L]
+- Project [id#0L, (id#0L % cast(100 as bigint)) AS value#1L]
   +- Range (0, 1000000, step=1, splits=Some(12))

== Optimized Logical Plan ==
Aggregate [value#1L], [value#1L, count(1) AS count#112L]
+- Project [(id#0L % 100) AS value#1L]
   +- Range (0, 1000000, step=1, splits=Some(12))

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[value#1L], functions=[count(1)], output=[value#1L, count#112L])
   +- Exchange hashpartitioning(value#1L, 36), ENSURE_REQUIREMENTS, [plan_id=552]
      +- HashAggregate(keys=[value#1L], functions=[partial_count(1)], output=[value#1L, count#117L])
         +- Project [(i

## Step 10 — Check partition distribution after shuffle

Skewed partitions can cause slow tasks even when the total data size is reasonable.

This example checks partition distribution after an aggregation.

In [10]:
partition_distribution = (
    agg_result
    .repartition(target_partitions)
    .withColumn("partition_id", spark_partition_id())
    .groupBy("partition_id")
    .count()
    .orderBy("partition_id")
)

partition_distribution.show(target_partitions, truncate=False)

+------------+-----+
|partition_id|count|
+------------+-----+
|0           |3    |
|1           |3    |
|2           |3    |
|3           |3    |
|4           |3    |
|5           |3    |
|6           |3    |
|7           |3    |
|8           |3    |
|9           |3    |
|10          |3    |
|11          |3    |
|12          |3    |
|13          |3    |
|14          |3    |
|15          |3    |
|16          |3    |
|17          |3    |
|18          |3    |
|19          |3    |
|20          |3    |
|21          |3    |
|22          |3    |
|23          |3    |
|24          |3    |
|25          |2    |
|26          |2    |
|27          |2    |
|28          |2    |
|29          |2    |
|30          |2    |
|31          |2    |
|32          |2    |
|33          |3    |
|34          |3    |
|35          |3    |
+------------+-----+



## Step 11 — Explain plan checklist

When reading `explain(True)`, look for:

- `BroadcastHashJoin` for broadcast joins
- `SortMergeJoin` for shuffle-heavy joins
- `Exchange` operators, which indicate shuffle
- pushed filters in scan nodes
- partition filters for partitioned datasets
- adaptive query execution changes

In [11]:
example_plan_df = (
    df_large
    .filter(col("value") < 10)
    .join(broadcast(df_small), on="id", how="inner")
    .groupBy("category")
    .count()
)

example_plan_df.explain(True)
example_plan_df.show()

== Parsed Logical Plan ==
'Aggregate ['category], ['category, 'count(1) AS count#140]
+- Project [id#0L, value#1L, category#85L]
   +- Join Inner, (id#0L = id#84L)
      :- Filter (value#1L < cast(10 as bigint))
      :  +- Project [id#0L, (id#0L % cast(100 as bigint)) AS value#1L]
      :     +- Range (0, 1000000, step=1, splits=Some(12))
      +- ResolvedHint (strategy=broadcast)
         +- Project [id#84L, (id#84L % cast(5 as bigint)) AS category#85L]
            +- Range (0, 100, step=1, splits=Some(12))

== Analyzed Logical Plan ==
category: bigint, count: bigint
Aggregate [category#85L], [category#85L, count(1) AS count#140L]
+- Project [id#0L, value#1L, category#85L]
   +- Join Inner, (id#0L = id#84L)
      :- Filter (value#1L < cast(10 as bigint))
      :  +- Project [id#0L, (id#0L % cast(100 as bigint)) AS value#1L]
      :     +- Range (0, 1000000, step=1, splits=Some(12))
      +- ResolvedHint (strategy=broadcast)
         +- Project [id#84L, (id#84L % cast(5 as bigint)) AS

## Step 12 — Quick reference summary

| Optimization | When to use | Impact |
|---|---|---|
| Partitioning | Before shuffle/write | High |
| Caching | DataFrame reused 2+ times | High |
| Broadcast Join | Small + large table join | Very High |
| Filter Pushdown | Queries with filters/joins | High |
| Column Pruning | Wide DataFrames | High |
| Shuffle Partitions | Aggregations and joins | Medium–High |
| AQE | Mixed-size workloads | Medium–High |
| Unpersist | After cached reuse is complete | Memory safety |

## Step 13 — Cleanup

For tutorial use, you can remove generated output. The Spark session is left running so you can continue experimenting.

Uncomment `spark.stop()` only when you are done with the notebook.

In [12]:
# safe_rm(BASE_PATH)

# Stop SparkSession when done.
# spark.stop()